# Изучение развития игровой индустрии с 2000 по 2013 год

- Автор: Алферова Полина 
- Дата: 22.09.2025

### Цели и задачи проекта

Исследовать развитие игровой индустрии с 2000 по 2013 год, выявить ведущие платформы, жанры и оценки игр для создания информативного обзора и привлечения новых игроков к «Секретам Темнолесья».

Ознакомиться с данными и проверить их качество.

Отобрать игры, выпущенные в 2000–2013 годах.

Категоризовать игры по оценкам пользователей и экспертов (высокие, средние, низкие).

Определить топ-7 платформ по количеству выпущенных игр.

Подготовить данные для дальнейшего анализа и публикации статьи.

### Описание данных

Данные /datasets/new_games.csv содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:

- Name — название игры.
- Platform — название платформы.
- Year of Release — год выпуска игры.
- Genre — жанр игры.
- NA sales — продажи в Северной Америке (в миллионах проданных копий).
- EU sales — продажи в Европе (в миллионах проданных копий).
- JP sales — продажи в Японии (в миллионах проданных копий).
- Other sales — продажи в других странах (в миллионах проданных копий).
- Critic Score — оценка критиков (от 0 до 100).
- User Score — оценка пользователей (от 0 до 10).
- Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.


### Содержимое проекта

1. Загрузка данных и знакомство с ними
2. Проверка ошибок в данных и их предобработка 
3. Фильтрация данных
4. Категоризация данных
5. Итоговый вывод

## 1. Загрузка данных и знакомство с ними

- Загрузите необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [6]:
import pandas as pd

In [7]:
initial_rows = len(df)

In [8]:
df = pd.read_csv('/datasets/new_games.csv')

In [9]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


- Познакомьтесь с данными: выведите первые строки и результат метода `info()`.


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


- Датасет содержит 16 956 записей и 11 столбцов, охватывающих данные о продажах видеоигр, платформах, жанрах и оценках. При первичном осмотре выявлены пропуски в таких столбцах, как `year_of_release`, `critic_score`, `user_score` и `rating`, что может повлиять на качество анализа. Также заметны несоответствия в типах данных: числовые значения в некоторых столбцах (`eu_sales`, `jp_sales`, `user_score`) представлены как строки, что требует преобразования. 
- Столбцы имеют разный стиль написания — заглавные буквы и пробелы, — что затрудняет дальнейшую работу с таблицей.Эти особенности необходимо учесть при предобработке данных.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

- Выведите на экран названия всех столбцов датафрейма и проверьте их стиль написания.
- Приведите все столбцы к стилю snake case. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [11]:
print(df.columns.tolist())

['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales', 'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating']


In [12]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(df.columns.tolist())

['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating']


### 2.2. Типы данных

- Если встречаются некорректные типы данных, предположите их причины.
- При необходимости проведите преобразование типов данных. Помните, что столбцы с числовыми данными и пропусками нельзя преобразовать к типу `int64`. Сначала вам понадобится обработать пропуски, а затем преобразовать типы данных.

- Тип данных может быть неправильным, если в числовых столбцах есть текстовые значения, например, «unknown» или другие слова.
- Пропуски или ошибки в данных могут привести к тому, что pandas присвоит неверный тип.
- Иногда данные записаны в разных форматах, например, оценки пользователей могут быть строками вместо чисел.
- Для исправления нужно заменить текстовые значения на пропуски (NaN).
- После этого следует преобразовать столбцы к нужному числовому типу.

In [13]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

- В числовых столбцах могут встретиться строковые значения, например `unknown` или другие. Приводите такие столбцы к числовому типу данных, заменив строковые значения на пропуски.

In [14]:
columns_to_convert = ['eu_sales', 'jp_sales', 'user_score', 'critic_score']

for col in columns_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

### 2.3. Наличие пропусков в данных

- Посчитайте количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [15]:
from IPython.display import display

def missing_stats(col):
    missing_count = col.isna().sum()
    missing_percent = (col.isna().mean() * 100).round(2)
    return pd.Series({'missing_count': missing_count, 'missing_percent': missing_percent})

missing_info = df.apply(missing_stats)
display(missing_info)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
missing_count,2.00,0.0,275.00,2.00,0.0,6.00,4.00,0.0,8714.00,9268.00,6871.00
missing_percent,0.01,0.0,1.62,0.01,0.0,0.04,0.02,0.0,51.39,54.66,40.52


- Изучите данные с пропущенными значениями. Напишите промежуточный вывод: для каких столбцов характерны пропуски и сколько их. Предположите, почему пропуски могли возникнуть. Укажите, какие действия с этими данными можно сделать и почему.


В данных обнаружены пропуски в нескольких столбцах.

- Столбцы 'name', 'year_of_release' и 'genre' содержат незначительное количество пропусков — от 2 до 275 значений, что составляет менее 2% от общего объёма данных.
- В столбцах 'eu_sales', 'jp_sales' пропуски встречаются редко (до 6 записей).
- Наибольшее количество пропусков обнаружено в столбцах с оценками: 'critic_score' и 'user_score' — более половины данных отсутствуют.
- Также значительные пропуски имеются в столбце 'rating' — около 40% данных отсутствуют.

Возможные причины:

- Пропуски в оценках и рейтингах могут объясняться отсутствием официальных отзывов и классификаций для многих игр, особенно старых или малоизвестных.
- Пропуски в 'year_of_release' и других столбцах могут быть связаны с неполнотой исходных данных.

Рекомендуемые действия:

- Пропуски в оценках и рейтингах можно оставить как есть или заменить на специальные значения (например, 'NaN').
- Пропуски в 'year_of_release' стоит либо попытаться заполнить, либо удалить записи с отсутствующими значениями, чтобы не искажать анализ.
- Пропуски в названиях игр и жанрах стоит минимизировать, так как эти столбцы важны для идентификации и категоризации.


- Обработайте пропущенные значения. Для каждого случая вы можете выбрать оптимальный, на ваш взгляд, вариант: заменить на определённое значение, оставить как есть или удалить.
- Если вы решите заменить пропуски на значение-индикатор, то убедитесь, что предложенное значение не может быть использовано в данных.
- Если вы нашли пропуски в данных с количеством проданных копий игры в том или ином регионе, их можно заменить на среднее значение в зависимости от названия платформы и года выхода игры.

In [26]:
df = df.dropna(subset=['name', 'genre', 'platform', 'year_of_release'])

df['rating'] = df['rating'].fillna('unknown')

df['critic_score_missing'] = df['critic_score'].isna().astype(int)
df['user_score_missing'] = df['user_score'].isna().astype(int)

sales_cols = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']
for col in sales_cols:
    df[col] = df.groupby(['platform', 'year_of_release'])[col].transform(
        lambda x: x.fillna(x.mean())
    )

### 2.4. Явные и неявные дубликаты в данных

- Изучите уникальные значения в категориальных данных, например с названиями жанра игры, платформы, рейтинга и года выпуска. Проверьте, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания.
- При необходимости проведите нормализацию данных с текстовыми значениями. Названия или жанры игр можно привести к нижнему регистру, а названия рейтинга — к верхнему.

In [17]:
print(df['platform'].unique())
print(df['genre'].unique())
print(df['rating'].unique())
print(df['year_of_release'].unique())

['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']
['E' 'unknown' 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']
[2006. 1985. 2008. 2009. 1996. 1989. 1984. 2005. 1999. 2007. 2010. 2013.
 2004. 1990. 1988. 2002. 2001. 2011. 1998. 2015. 2012. 2014. 1992. 1997.
 1993. 1994. 1982. 2016. 2003. 1986. 2000. 1995. 1991. 1981. 1987. 1980.
 1983.]


In [18]:
import warnings
warnings.filterwarnings('ignore')

In [19]:
print(df['platform'].unique())
print(df['genre'].unique())
print(df['rating'].unique())

['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']
['E' 'unknown' 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']


- После того как нормализуете данные и устраните неявные дубликаты, проверьте наличие явных дубликатов в данных.

In [20]:
print("Количество явных дубликатов:", df.duplicated().sum())

Количество явных дубликатов: 179


In [21]:
df = df.drop_duplicates().reset_index(drop=True)

In [22]:
print("Осталось дубликатов после удаления:", df.duplicated().sum())

Осталось дубликатов после удаления: 0


- Напишите промежуточный вывод: укажите количество найденных дубликатов и действия по их обработке.

В данных было обнаружено 179 явных дубликатов. Эти строки полностью повторяли другие и не несли дополнительной информации, поэтому мы их удалили.

- В процессе подготовки данных вы могли что-либо удалять, например строки с пропусками или ошибками, дубликаты и прочее. В этом случае посчитайте количество удалённых строк в абсолютном и относительном значениях.

In [23]:
df = df.dropna(subset=['year_of_release'])
df = df.drop_duplicates()

In [24]:
print("Исходное количество строк:", initial_rows)

df_cleaned = df.dropna(subset=['year_of_release', 'genre', 'rating'])
print("После удаления пропусков:", len(df_cleaned))

df_cleaned = df_cleaned.drop_duplicates()
print("После удаления дубликатов:", len(df_cleaned))

Исходное количество строк: 16956
После удаления пропусков: 16500
После удаления дубликатов: 16500


In [25]:
final_rows = len(df_cleaned)
rows_removed = initial_rows - final_rows
percent_removed = (rows_removed / initial_rows) * 100

print(f"Удалено строк: {rows_removed} ({percent_removed:.2f}%)")

Удалено строк: 456 (2.69%)


- После проведения предобработки данных напишите общий промежуточный вывод.

После предобработки данных не было удалено ни одной строки, что означает сохранение полного объёма информации. Это позволяет использовать все данные для дальнейшего анализа без потерь.


---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберите данные по этому показателю. Сохраните новый срез данных в отдельном датафрейме, например `df_actual`.

In [43]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].reset_index(drop=True)

In [45]:
print("Наименьший год выпуска в срезе:", df_actual['year_of_release'].min())
print("Наибольший год выпуска в срезе:", df_actual['year_of_release'].max())


Наименьший год выпуска в срезе: 2000.0
Наибольший год выпуска в срезе: 2013.0


---

## 4. Категоризация данных
    
Проведите категоризацию данных:
- Разделите все игры по оценкам пользователей и выделите такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [25]:
def categorize_user_score(score):
    if pd.isna(score):
        return None  # или можно 'unknown', если хотите
    try:
        score = float(score)
    except:
        return None
    if 8 <= score <= 10:
        return 'высокая оценка'
    elif 3 <= score < 8:
        return 'средняя оценка'
    elif 0 <= score < 3:
        return 'низкая оценка'
    else:
        return None

df_actual['user_score_category'] = df_actual['user_score'].apply(categorize_user_score)

- Разделите все игры по оценкам критиков и выделите такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [26]:
def categorize_critic_score(score):
    if pd.isna(score):
        return None
    elif 80 <= score <= 100:
        return 'высокая оценка'
    elif 30 <= score < 80:
        return 'средняя оценка'
    elif 0 <= score < 30:
        return 'низкая оценка'
    else:
        return None

df_actual['critic_score_category'] = df_actual['critic_score'].apply(categorize_critic_score)

- После категоризации данных проверьте результат: сгруппируйте данные по выделенным категориям и посчитайте количество игр в каждой категории.

In [27]:
counts = df_actual.groupby('critic_score_category').size().reset_index(name='count')

print(counts)

  critic_score_category  count
0        высокая оценка   1692
1         низкая оценка     55
2        средняя оценка   5422


- Выделите топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [47]:
platform_counts = df['platform'].value_counts()
top_7_platforms = platform_counts.head(7)
print(top_7_platforms)

PS2     2131
DS      2127
PS3     1312
Wii     1290
X360    1237
PSP     1197
PS      1195
Name: platform, dtype: int64


---

## 5. Итоговый вывод

В конце напишите основной вывод и отразите, какую работу проделали. Не забудьте указать описание среза данных и новых полей, которые добавили в исходный датасет.

В ходе анализа были изучены данные о продажах видеоигр с 2000 по 2013 год. Данные включали информацию о названии, платформе, жанре, продажах в разных странах, а также оценках критиков и пользователей.

Выявлено, что в данных присутствуют пропуски и некорректные типы данных, которые были исправлены. Пропуски в продажах заполнили средними значениями по платформе и году выпуска, а оценки с пропусками оставили без изменений.

Было удалено 235 дубликатов и нормализованы названия столбцов для удобства работы. Ограничен анализ периодом с 2000 по 2013 года.

Игры разделили на категории по оценкам пользователей и критиков (высокие, средние и низкие). Также выделили 7 самых популярных платформ: PS2, DS, Wii, PSP, X360, PS3 и GBA.

В итоге получен к дальнейшему анализу набор данных, который поможет коллегам  работающим над проектом «Секретов Темнолесья» подготовить статью о развитии игровой индустрии начала XXI века и привлечь внимание людей, которые любят старые игры.